**Checkpointing - Automatically save execution state so crews, flows, and agents can resume after failures.**

**Checkpointing saves a snapshot of execution state during a run so a crew, flow, or agent can resume after a failure or be forked into an alternate branch.**

In [1]:
import os
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, LLM, CheckpointConfig

load_dotenv()
api_key=os.getenv("OPENAI_API_KEY")

llm = LLM(model="gpt-4o-mini", temperature=0)

def make_crew(checkpoint=None, agent_checkpoint=None):
    teacher = Agent(
        role="AI Teacher",
        goal="Explain concepts simply",
        backstory="You teach beginners using short explanations.",
        llm=llm,
        checkpoint=agent_checkpoint,
        verbose=False
    )

    explain = Task(
        description="Explain artificial intelligence in one sentence.",
        expected_output="One simple sentence.",
        agent=teacher
    )

    example = Task(
        description="Give one everyday example of artificial intelligence.",
        expected_output="One short example.",
        agent=teacher
    )

    return Crew(
        agents=[teacher],
        tasks=[explain, example],
        checkpoint=checkpoint,
        verbose=False
    )

**ENABLE CHECKPOINT WITH DEFAULT**

In [2]:
crew = make_crew(checkpoint=True)

result = await crew.kickoff_async()
print(result.raw)

One everyday example of artificial intelligence is a virtual assistant, like Siri or Google Assistant. These programs can understand your voice commands, answer questions, set reminders, and help you with various tasks, mimicking human-like conversation and problem-solving.


**CUSTOMIZE STORAGE AND FREQUENCY**

In [3]:
crew = make_crew(
    checkpoint=CheckpointConfig(
        location="./custom_checkpoints",
        on_events=["task_completed"],
        max_checkpoints=5
    )
)

result = await crew.kickoff_async()
print(result.raw)

One everyday example of artificial intelligence is a virtual assistant, like Siri or Google Assistant. These programs can understand your voice commands, answer questions, and help you with tasks like setting reminders or playing music, all by using AI to process language and make decisions.


**CHOOSE A STORAGE PROVIDER - JSON FILES**

In [4]:
from crewai.state import JsonProvider

crew = make_crew(
    checkpoint=CheckpointConfig(
        location="./json_checkpoints",
        provider=JsonProvider()
    )
)

result = await crew.kickoff_async()
print(result.raw)

One everyday example of artificial intelligence is a virtual assistant, like Siri or Google Assistant. These programs can understand your voice commands, answer questions, set reminders, and help you with tasks, mimicking human-like conversation and problem-solving.


**OPT ONE AGENT OUT**

In [5]:
crew = make_crew(
    checkpoint=True,
    agent_checkpoint=False
)

result = await crew.kickoff_async()
print(result.raw)

One everyday example of artificial intelligence is a virtual assistant, like Siri or Google Assistant. These programs can understand your voice commands, answer questions, and help you with tasks like setting reminders or playing music, all by using AI to process language and make decisions.


**CHECKPOINT A CREW**

In [7]:
crew = make_crew(
    checkpoint=CheckpointConfig(location="./crew_checkpoints")
)

result = await crew.kickoff_async()
print(result.raw)

One everyday example of artificial intelligence is a virtual assistant, like Siri or Alexa. These assistants can understand your voice commands, answer questions, play music, and control smart home devices, mimicking human-like conversation and decision-making.


**CHECKPOINT A FLOW**

In [8]:
from crewai.flow.flow import Flow, start, listen

class GreetingFlow(Flow):

    @start()
    def get_name(self):
        self.state["name"] = "John"

    @listen(get_name)
    def greet(self):
        return f"Hello, {self.state['name']}!"

flow = GreetingFlow(
    checkpoint=CheckpointConfig(
        location="./flow_checkpoints",
        on_events=["method_execution_finished"]
    )
)

result = await flow.kickoff_async()
print(result)

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: GreetingFlow                                                                                             │
│  ID: 16fd6fb8-d1a2-4b73-97a5-6bb1d534bf9f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: GreetingFlow                                                                                             │
│  ID: 16fd6fb8-d1a2-4b73-97a5-6bb1d534bf9f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: get_name                                                                                               │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: get_name                                                                                               │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: greet                                                                                                  │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: greet                                                                                                  │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: GreetingFlow                                                                                             │
│  ID: 16fd6fb8-d1a2-4b73-97a5-6bb1d534bf9f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Hello, John!


**CHECKPOINT A AGENT**

In [9]:
agent = Agent(
    role="AI Teacher",
    goal="Explain concepts simply",
    backstory="You teach beginners.",
    llm=llm,
    checkpoint=CheckpointConfig(
        location="./agent_checkpoints",
        on_events=["lite_agent_execution_completed"]
    )
)

result = await agent.kickoff_async(
    "Explain artificial intelligence in one sentence."
)
print(result.raw)

Artificial intelligence is the ability of a computer or machine to perform tasks that typically require human intelligence, such as understanding language, recognizing patterns, and making decisions.
